# RSS Feed Timestamp Testing

Tests whether RSS feeds provide full datetime timestamps (not just dates),
and whether we can filter feed entries to only the last N hours.

**Goal:** Determine if we can use RSS timestamps for hour-level lookback filtering
(e.g., "only entries from the last 1.5 hours").

In [8]:
!uv pip install pandas

Resolved 4 packages in 271ms                                         
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)-------------------     0 B/9.50 MiB            
⠙ Preparing packages... (0/2)------------------- 16.00 KiB/9.50 MiB          
⠙ Preparing packages... (0/2)------------------- 32.00 KiB/9.50 MiB          
⠙ Preparing packages... (0/2)------------------- 48.00 KiB/9.50 MiB          
⠙ Preparing packages... (0/2)------------------- 63.96 KiB/9.50 MiB          
⠙ Preparing packages... (0/2)------------------- 79.96 KiB/9.50 MiB          
⠙ Preparing packages... (0/2)------------------- 95.96 KiB/9.50 MiB          
⠙ Preparing packages... (0/2)------------------- 95.96 KiB/9.50 MiB          
numpy                ------------------------------     0 B/5.08 MiB
⠙ Preparing packages... (0/2)------------------- 95.96 KiB/9.50 MiB          
numpy                ------------------------------ 14.81 KiB/5.08 MiB
⠙ Prepa

In [9]:
import json
import feedparser
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pandas as pd

REGISTRY_PATH = Path("../discovery/output/registry_export_20260716_011537.json")

# How many hours back to filter
LOOKBACK_HOURS = 1.5

# How many feeds to test
MAX_FEEDS = 8

In [10]:
# Load registry and pick a diverse set of RSS feeds
with open(REGISTRY_PATH) as f:
    registry = json.load(f)

# Collect unique RSS URLs across different platforms
seen_urls = set()
seen_platforms = set()
test_feeds = []

for team in registry:
    for blog in team.get("blogs", []):
        rss_url = blog.get("rss_url", "")
        platform = blog.get("platform", "unknown")
        if (
            rss_url
            and rss_url not in seen_urls
            and blog.get("accessible")
            and blog.get("recency_status") == "active"
            and platform not in seen_platforms
        ):
            seen_urls.add(rss_url)
            seen_platforms.add(platform)
            test_feeds.append({
                "team": team["team"],
                "rss_url": rss_url,
                "platform": platform,
            })
            if len(test_feeds) >= MAX_FEEDS:
                break
    if len(test_feeds) >= MAX_FEEDS:
        break

print(f"Selected {len(test_feeds)} feeds from different platforms:\n")
for i, f in enumerate(test_feeds, 1):
    print(f"  {i}. {f['team']:30s} | {f['platform'][:45]}")
    print(f"     {f['rss_url']}")

Selected 8 feeds from different platforms:

  1. Abilene Christian Wildcats     | Official Athletics Site (Sidearm Sports)
     https://acusports.com/rss?path=mbball
  2. Air Force Falcons              | Official Athletic Site (Sidearm Sports)
     https://goairforcefalcons.com/rss?path=mbball
  3. Akron Zips                     | Substack
     https://akronzippedup.substack.com/feed
  4. Akron Zips                     | Official Athletics News Archive (Sidearm Spor
     https://gozips.com/rss?path=mbball
  5. Alabama A&M Bulldogs           | WordPress – HBCU Gameday Blog (Alabama A&M ta
     https://hbcugameday.com/tag/alabama-am-bulldogs/feed/
  6. Alabama Crimson Tide           | XenForo
     https://www.tidefans.com/forums/basketball.4/index.rss
  7. Alabama Crimson Tide           | Blog / News (TideFans)
     https://news.tidefans.com/feed/
  8. Alabama Crimson Tide           | FanSided Blog
     https://bamahammer.com/feed/


## Fetch feeds and inspect raw timestamps

For each feed, we parse it and look at what timestamp data feedparser gives us:
- `published_parsed` / `updated_parsed` — `time.struct_time` with full datetime
- `published` / `updated` — raw string from the feed

In [11]:
def parse_entry_datetime(entry) -> dict:
    """Extract all timestamp info from a feedparser entry."""
    result = {
        "title": entry.get("title", "")[:80],
        "link": entry.get("link", ""),
        "published_raw": entry.get("published", ""),
        "updated_raw": entry.get("updated", ""),
        "published_parsed": None,
        "updated_parsed": None,
        "best_datetime": None,
        "has_time_component": False,
    }

    # Try published_parsed first, then updated_parsed
    for field in ("published_parsed", "updated_parsed"):
        ts = entry.get(field)
        if ts:
            try:
                dt = datetime(*ts[:6], tzinfo=timezone.utc)
                result[field] = dt.isoformat()
                if result["best_datetime"] is None:
                    result["best_datetime"] = dt
                # Check if time is midnight (likely date-only)
                if dt.hour != 0 or dt.minute != 0 or dt.second != 0:
                    result["has_time_component"] = True
            except (TypeError, ValueError):
                continue

    return result


# Fetch and parse all feeds
feed_results = []

for feed_info in test_feeds:
    print(f"\n{'='*80}")
    print(f"Fetching: {feed_info['team']} ({feed_info['platform']})")
    print(f"URL: {feed_info['rss_url']}")
    print(f"{'='*80}")

    feed = feedparser.parse(
        feed_info["rss_url"],
        agent="Mozilla/5.0 (compatible; BlogSearchBot/1.0)",
    )

    if hasattr(feed, "status"):
        print(f"  HTTP Status: {feed.status}")
    if feed.bozo:
        print(f"  ⚠️  Parse warning: {feed.bozo_exception}")

    entries_data = []
    for entry in feed.entries[:5]:  # First 5 entries per feed
        parsed = parse_entry_datetime(entry)
        entries_data.append(parsed)
        
        time_icon = "🕐" if parsed["has_time_component"] else "📅"
        print(f"\n  {time_icon} {parsed['title'][:60]}")
        print(f"     published_raw:    {parsed['published_raw']}")
        print(f"     updated_raw:      {parsed['updated_raw']}")
        print(f"     published_parsed: {parsed['published_parsed']}")
        print(f"     updated_parsed:   {parsed['updated_parsed']}")
        print(f"     has_time:         {parsed['has_time_component']}")

    feed_results.append({
        "feed_info": feed_info,
        "entries": entries_data,
        "total_entries": len(feed.entries),
    })


Fetching: Abilene Christian Wildcats (Official Athletics Site (Sidearm Sports))
URL: https://acusports.com/rss?path=mbball
  HTTP Status: 200

  🕐 Tanner Announces 2026-27 Coaching Staff
     published_raw:    Fri, 17 Jul 2026 10:48:00 CST
     updated_raw:      Fri, 17 Jul 2026 10:48:00 CST
     published_parsed: 2026-07-17T16:48:00+00:00
     updated_parsed:   2026-07-17T16:48:00+00:00
     has_time:         True

  🕐 Two Wildcats Earn NABC Academic Honor
     published_raw:    Wed, 15 Jul 2026 16:00:00 CST
     updated_raw:      Wed, 15 Jul 2026 16:00:00 CST
     published_parsed: 2026-07-15T22:00:00+00:00
     updated_parsed:   2026-07-15T22:00:00+00:00
     has_time:         True

  🕐 UAC Unveils Conference Schedules For Men’s, Women’s Hoops
     published_raw:    Wed, 15 Jul 2026 11:00:00 CST
     updated_raw:      Wed, 15 Jul 2026 11:00:00 CST
     published_parsed: 2026-07-15T17:00:00+00:00
     updated_parsed:   2026-07-15T17:00:00+00:00
     has_time:         True

  🕐 All-T

## Summary: Which feeds have real timestamps vs date-only?

In [12]:
print(f"\n{'='*80}")
print("TIMESTAMP SUPPORT SUMMARY")
print(f"{'='*80}\n")

summary_rows = []
for result in feed_results:
    entries_with_time = sum(1 for e in result["entries"] if e["has_time_component"])
    entries_total = len(result["entries"])
    has_any_time = entries_with_time > 0

    summary_rows.append({
        "Team": result["feed_info"]["team"],
        "Platform": result["feed_info"]["platform"][:35],
        "Entries Checked": entries_total,
        "Has Time Component": f"{entries_with_time}/{entries_total}",
        "Hour-level Filter?": "✅ YES" if has_any_time else "❌ Date-only",
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

feeds_with_time = sum(1 for r in summary_rows if "YES" in r["Hour-level Filter?"])
print(f"\n→ {feeds_with_time}/{len(summary_rows)} feeds support hour-level timestamp filtering")


TIMESTAMP SUPPORT SUMMARY

                      Team                            Platform  Entries Checked Has Time Component Hour-level Filter?
Abilene Christian Wildcats Official Athletics Site (Sidearm Sp                5                5/5              ✅ YES
         Air Force Falcons Official Athletic Site (Sidearm Spo                5                5/5              ✅ YES
                Akron Zips                            Substack                5                5/5              ✅ YES
                Akron Zips Official Athletics News Archive (Si                5                5/5              ✅ YES
      Alabama A&M Bulldogs WordPress – HBCU Gameday Blog (Alab                2                2/2              ✅ YES
      Alabama Crimson Tide                             XenForo                0                0/0        ❌ Date-only
      Alabama Crimson Tide              Blog / News (TideFans)                5                5/5              ✅ YES
      Alabama Crimson Tide  

## Apply lookback filter (last 1.5 hours)

Now let's apply the actual time-based filter to see which entries would pass.

In [13]:
now = datetime.now(timezone.utc)
cutoff = now - timedelta(hours=LOOKBACK_HOURS)

print(f"Current time (UTC):  {now.isoformat()}")
print(f"Lookback:            {LOOKBACK_HOURS} hours")
print(f"Cutoff time (UTC):   {cutoff.isoformat()}")
print(f"\nOnly entries published AFTER the cutoff will pass the filter.\n")
print("=" * 80)

total_entries = 0
passed_entries = 0
skipped_no_timestamp = 0

for result in feed_results:
    feed_info = result["feed_info"]
    print(f"\n📡 {feed_info['team']} ({feed_info['platform'][:30]})")

    for entry in result["entries"]:
        total_entries += 1
        dt = entry["best_datetime"]

        if dt is None:
            skipped_no_timestamp += 1
            print(f"   ⚠️  NO TIMESTAMP — {entry['title'][:50]}")
            continue

        if dt >= cutoff:
            passed_entries += 1
            age_minutes = (now - dt).total_seconds() / 60
            print(f"   ✅ PASS ({age_minutes:.0f} min ago) — {entry['title'][:50]}")
        else:
            age_hours = (now - dt).total_seconds() / 3600
            if entry["has_time_component"]:
                print(f"   ❌ FAIL ({age_hours:.1f}h ago) — {entry['title'][:50]}")
            else:
                # Date-only: midnight UTC might be misleading
                print(f"   ❌ FAIL ({age_hours:.1f}h ago, date-only⚠️) — {entry['title'][:50]}")

print(f"\n{'='*80}")
print(f"FILTER RESULTS:")
print(f"  Total entries checked:    {total_entries}")
print(f"  Passed (within {LOOKBACK_HOURS}h):    {passed_entries}")
print(f"  Failed (too old):         {total_entries - passed_entries - skipped_no_timestamp}")
print(f"  Skipped (no timestamp):   {skipped_no_timestamp}")

Current time (UTC):  2026-07-24T18:54:24.723554+00:00
Lookback:            1.5 hours
Cutoff time (UTC):   2026-07-24T17:24:24.723554+00:00

Only entries published AFTER the cutoff will pass the filter.


📡 Abilene Christian Wildcats (Official Athletics Site (Sidea)
   ❌ FAIL (170.1h ago) — Tanner Announces 2026-27 Coaching Staff
   ❌ FAIL (212.9h ago) — Two Wildcats Earn NABC Academic Honor
   ❌ FAIL (217.9h ago) — UAC Unveils Conference Schedules For Men’s, Women’
   ❌ FAIL (241.4h ago) — All-Time Greats Headline ACU Sports Hall of Fame C
   ❌ FAIL (555.8h ago) — ACU Welcomes New Members in Officially Rebranded U

📡 Air Force Falcons (Official Athletic Site (Sidear)
   ❌ FAIL (1104.9h ago) — Tanner Massey Joins Air Force Men’s Basketball Sta
   ❌ FAIL (1394.3h ago) — Air Force MBB 2025-26 Season Recap
   ❌ FAIL (2039.9h ago) — Air Force Men’s Basketball Announces 2026 Program 
   ❌ FAIL (2520.9h ago) — Jon Jordan and Capt. Sid Tomes Remain On Air Force
   ❌ FAIL (2544.9h ago) — Michae

## Conclusion

Key findings:
- Feeds with `has_time_component=True` → safe to filter at hour-level granularity
- Feeds with date-only timestamps (midnight UTC) → hour-level filter unreliable;
  for these you'd need to include entries from the entire day or fall back to date-level filtering

**Recommended approach for `lookback_hours` in workflow mode:**
1. If entry has a real time component → apply exact hour-level cutoff
2. If entry is date-only (00:00:00 UTC) → include if the date falls within the lookback window
   (i.e., treat as "published sometime that day")